In [8]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table

In [9]:
df = pd.read_csv("../../data/OLF_Reason Not Working.csv")

In [10]:
df_final = df.melt(
    id_vars="reason_not_working",
    var_name="year",
    value_name="total_outside_labor_force"
)
df_final["total_outside_labor_force"] = df_final["total_outside_labor_force"]*1000
df_final = df_final.sort_values(["year", "reason_not_working"]).reset_index(drop=True)
df_final = df_final[["year", "reason_not_working", "total_outside_labor_force"]]
df_final.head(20)

,year,reason_not_working,total_outside_labor_force
0,2016,Disabled,8400.0
1,2016,Going for further studies,15700.0
2,2016,Housework/ family responsibilities,212500.0
3,2016,Not interested/ just completed study,22500.0
4,2016,Retired/ Old age,156000.0
5,2016,Schooling/ training programme,244500.0
6,2017,Disabled,12000.0
7,2017,Going for further studies,12300.0
8,2017,Housework/ family responsibilities,245900.0
9,2017,Not interested/ just completed study,22900.0


In [11]:
write_table(df_final, "sc_bronze", "dosm_olf_reason")

Table sc_bronze.dosm_olf_reason written successfully.


In [12]:
def transform_olf_by_age(file_path):
    df = pd.read_csv(file_path)

    # 1. Pivot dataframe
    df_long = df.melt(
        id_vars=["statistics"],
        var_name="year",
        value_name="value"
    )

    # 2. Parse statistics into age_group and qualification
    def parse_stats(stat_name):
        if "_" in stat_name:
            age_part, qual_part = stat_name.split("_", 1)
            return age_part.strip(), qual_part.strip()
        return "Total", stat_name.strip()

    df_long[['age_group', 'qual_cat']] = df_long['statistics'].apply(
        lambda x: pd.Series(parse_stats(x))
    )

    # 3. Extract qualification from qual_cat
    def extract_qualification(qual_cat):
        if pd.isna(qual_cat):
            return None
        if "_" in qual_cat:
            return qual_cat.split("_", 1)[1]
        return qual_cat

    df_long['qualification'] = df_long['qual_cat'].apply(extract_qualification)

    # 4. Convert numeric and scale
    df_long['total_outside_labor_force'] = pd.to_numeric(df_long['value'], errors='coerce')
    df_long = df_long.dropna(subset=['total_outside_labor_force'])
    df_long['total_outside_labor_force'] = df_long['total_outside_labor_force'] * 1000

    df_final = df_long[['year', 'age_group', 'qualification', 'total_outside_labor_force']].copy()
    df_final = df_final.sort_values(['year', 'age_group', 'qualification']).reset_index(drop=True)

    return df_final

In [13]:
df2 = transform_olf_by_age("../../data/OLF_Age group.csv")
df2.head(20)

,year,age_group,qualification,total_outside_labor_force
0,2016,25 - 34,degree,83700.0
1,2016,25 - 34,diploma,95000.0
2,2016,35 - 44,degree,25800.0
3,2016,35 - 44,diploma,34400.0
4,2016,≤ 24,degree,31600.0
5,2016,≤ 24,diploma,184700.0
6,2016,≥ 45,degree,99700.0
7,2016,≥ 45,diploma,104700.0
8,2017,25 - 34,degree,94300.0
9,2017,25 - 34,diploma,100200.0


In [14]:
write_table(df2, "sc_bronze", "dosm_olf_age")

Table sc_bronze.dosm_olf_age written successfully.
